In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from google.colab import files

existing = list(Path("/content").glob("Day14_Food_Delivery_Visualization_Dataset.csv"))

if existing:
    file_path = str(existing[0])
else:
    uploaded = files.upload()
    file_path = next(iter(uploaded))

df = pd.read_csv(file_path)
df["Date"] = pd.to_datetime(df["Date"])
df.head()

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate Rows:", df.duplicated().sum())

display(df.describe().T)

In [ ]:
daily = df.groupby("Date").agg(
    Orders=("Orders","sum"),
    Revenue=("Revenue","sum")
).reset_index()

fig, ax1 = plt.subplots(figsize=(12,5))
ax1.plot(daily["Date"], daily["Orders"], marker="o", label="Orders")
ax1.set_title("Orders Over Time")
ax1.set_xlabel("Date")
ax1.set_ylabel("Orders")
ax1.tick_params(axis="x", rotation=45)
ax1.legend()
plt.tight_layout()
plt.show()

print("Interpretation: The line plot shows how order volume changes over time, helping identify high-demand and low-demand periods.")

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(daily["Date"], daily["Revenue"], marker="o", label="Revenue")
ax.set_title("Revenue Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print("Interpretation: Revenue generally moves with business activity, with peaks indicating stronger-performing dates.")

In [ ]:
city_summary = df.groupby("City")["Revenue"].sum().sort_values(ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(x=city_summary.values, y=city_summary.index)
plt.title("Total Revenue by City")
plt.xlabel("Total Revenue")
plt.ylabel("City")
plt.tight_layout()
plt.show()

print("Interpretation: The bars compare total revenue across cities and reveal which locations contribute most to the business.")

In [ ]:
cuisine_summary = df.groupby("Cuisine")["Orders"].sum().sort_values(ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(x=cuisine_summary.values, y=cuisine_summary.index)
plt.title("Total Orders by Cuisine")
plt.xlabel("Total Orders")
plt.ylabel("Cuisine")
plt.tight_layout()
plt.show()

print("Interpretation: This chart identifies the cuisines with the highest order demand.")

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Marketing_Spend", y="Revenue", hue="City", s=70)
plt.title("Marketing Spend vs Revenue")
plt.xlabel("Marketing Spend")
plt.ylabel("Revenue")
plt.legend(title="City", bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.show()

corr_marketing = df[["Marketing_Spend","Revenue"]].corr().iloc[0,1]
print(f"Interpretation: Marketing spend and revenue have a Pearson correlation of {corr_marketing:.2f}. The scatter pattern shows whether higher marketing investment is associated with higher revenue.")

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Avg_Delivery_Minutes", y="Customer_Rating", hue="Weather", s=70)
plt.title("Delivery Time vs Customer Rating")
plt.xlabel("Average Delivery Time (Minutes)")
plt.ylabel("Customer Rating")
plt.legend(title="Weather", bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.show()

corr_delivery = df[["Avg_Delivery_Minutes","Customer_Rating"]].corr().iloc[0,1]
print(f"Interpretation: Delivery time and customer rating have a Pearson correlation of {corr_delivery:.2f}. A downward pattern would indicate that slower deliveries are associated with lower ratings.")

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["Orders"], bins=20, kde=True)
plt.title("Distribution of Orders")
plt.xlabel("Orders")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Interpretation: The histogram shows the typical order volume and the spread of order counts across observations.")

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x="City", y="Revenue")
plt.title("Revenue Distribution by City")
plt.xlabel("City")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Interpretation: Box plots compare the median, spread, and potential outliers in revenue across cities.")

In [ ]:
plt.figure(figsize=(9,5))
sns.violinplot(data=df, x="Cuisine", y="Average_Order_Value")
plt.title("Average Order Value Distribution by Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("Average Order Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Interpretation: Violin plots show both the distribution shape and concentration of average order values for each cuisine.")

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="Order_Channel", hue="Weather")
plt.title("Order Channel Counts by Weather")
plt.xlabel("Order Channel")
plt.ylabel("Number of Records")
plt.legend(title="Weather")
plt.tight_layout()
plt.show()

print("Interpretation: This count plot compares the use of different order channels under different weather conditions.")

In [ ]:
weather_summary = df.groupby("Weather").agg(
    Orders=("Orders","mean"),
    Revenue=("Revenue","mean"),
    Customer_Rating=("Customer_Rating","mean")
).sort_values("Revenue", ascending=False)

display(weather_summary.round(2))

plt.figure(figsize=(8,5))
sns.barplot(data=weather_summary.reset_index(), x="Weather", y="Revenue")
plt.title("Average Revenue by Weather")
plt.xlabel("Weather")
plt.ylabel("Average Revenue")
plt.tight_layout()
plt.show()

print("Interpretation: The chart compares average revenue under different weather conditions and highlights weather-related performance differences.")

In [ ]:
channel_summary = df.groupby("Order_Channel").agg(
    Orders=("Orders","mean"),
    Revenue=("Revenue","mean"),
    Average_Order_Value=("Average_Order_Value","mean"),
    Customer_Rating=("Customer_Rating","mean")
).sort_values("Revenue", ascending=False)

display(channel_summary.round(2))

plt.figure(figsize=(8,5))
sns.barplot(data=channel_summary.reset_index(), x="Order_Channel", y="Revenue")
plt.title("Average Revenue by Order Channel")
plt.xlabel("Order Channel")
plt.ylabel("Average Revenue")
plt.tight_layout()
plt.show()

print("Interpretation: This comparison shows which order channels generate stronger average revenue.")

In [ ]:
numeric_cols = [
    "Orders", "Average_Order_Value", "Revenue", "Marketing_Spend",
    "Discounts", "Avg_Delivery_Minutes", "Customer_Rating",
    "Repeat_Customer_Percent"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Food Delivery Metrics")
plt.tight_layout()
plt.show()

display(corr.round(3))

In [ ]:
city_cuisine = df.pivot_table(
    index="City",
    columns="Cuisine",
    values="Revenue",
    aggfunc="mean"
)

plt.figure(figsize=(11,6))
sns.heatmap(city_cuisine, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Average Revenue by City and Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("City")
plt.tight_layout()
plt.show()

print("Interpretation: This heatmap identifies combinations of city and cuisine that generate relatively higher or lower average revenue.")

In [ ]:
promotion_summary = df.assign(
    Promotion=df["Discounts"].gt(0).map({True:"Discount Applied", False:"No Discount"})
).groupby("Promotion").agg(
    Orders=("Orders","mean"),
    Revenue=("Revenue","mean"),
    Customer_Rating=("Customer_Rating","mean")
)

display(promotion_summary.round(2))

plt.figure(figsize=(7,5))
sns.barplot(data=promotion_summary.reset_index(), x="Promotion", y="Revenue")
plt.title("Average Revenue: Discount vs No Discount")
plt.xlabel("Promotion Status")
plt.ylabel("Average Revenue")
plt.tight_layout()
plt.show()

print("Interpretation: This comparison examines whether observations with discounts have different average revenue from those without discounts.")

In [ ]:
# Data-driven summary
marketing_corr = df["Marketing_Spend"].corr(df["Revenue"])
delivery_corr = df["Avg_Delivery_Minutes"].corr(df["Customer_Rating"])
orders_revenue_corr = df["Orders"].corr(df["Revenue"])
repeat_revenue_corr = df["Repeat_Customer_Percent"].corr(df["Revenue"])

best_city = df.groupby("City")["Revenue"].mean().idxmax()
best_city_val = df.groupby("City")["Revenue"].mean().max()
best_cuisine = df.groupby("Cuisine")["Orders"].sum().idxmax()
best_cuisine_orders = df.groupby("Cuisine")["Orders"].sum().max()
best_channel = df.groupby("Order_Channel")["Revenue"].mean().idxmax()
best_channel_rev = df.groupby("Order_Channel")["Revenue"].mean().max()
best_weather = df.groupby("Weather")["Revenue"].mean().idxmax()
best_weather_rev = df.groupby("Weather")["Revenue"].mean().max()

print("KEY FINDINGS")
findings = [
    f"1. {best_city} had the highest average revenue per observation among cities, at {best_city_val:,.2f}.",
    f"2. {best_cuisine} recorded the highest total number of orders, with {best_cuisine_orders:,.0f} orders across the dataset.",
    f"3. Marketing spend and revenue had a Pearson correlation of {marketing_corr:.2f}, indicating the strength of their linear association.",
    f"4. Delivery time and customer rating had a Pearson correlation of {delivery_corr:.2f}; the sign indicates whether longer delivery times are associated with lower or higher ratings.",
    f"5. Orders and revenue had a Pearson correlation of {orders_revenue_corr:.2f}, showing how strongly order volume is associated with revenue.",
    f"6. {best_channel} had the highest average revenue among order channels, at {best_channel_rev:,.2f}.",
    f"7. {best_weather} produced the highest average revenue among weather categories, at {best_weather_rev:,.2f}.",
    f"8. Repeat-customer percentage and revenue had a Pearson correlation of {repeat_revenue_corr:.2f}, providing an indication of the relationship between customer retention and revenue."
]

for f in findings:
    print(f)

In [ ]:
# Final portfolio summary
print("Dataset:", df.shape[0], "rows and", df.shape[1], "columns")
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Cities:", df["City"].nunique())
print("Cuisines:", df["Cuisine"].nunique())
print("Order channels:", df["Order_Channel"].nunique())
print("Weather categories:", df["Weather"].nunique())

print("\nOverall averages:")
display(df[[
    "Orders", "Average_Order_Value", "Revenue", "Marketing_Spend",
    "Discounts", "Avg_Delivery_Minutes", "Customer_Rating",
    "Repeat_Customer_Percent"
]].mean().to_frame("Mean").round(2))